In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # add project root so 'src' is importable

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

In [ ]:
from src.anomaly_detection import (
    detect_profile_shape_anomaly,
    detect_trend_anomaly,
    detect_event_response_anomaly,
)

df = pd.read_parquet("../data/processed/dataset_finale_2023-09-01_2025-09-30.parquet")
df = df[~df["chiave"].isin([41, 251])]
df = df[df["accuratezza"].notna()]

r1 = detect_profile_shape_anomaly(df)
r2 = detect_trend_anomaly(df)
r3 = detect_event_response_anomaly(df)

print("Anomaly 1 — shape:", r1.shape, "| flagged:", r1["is_anomaly"].sum())
print("Anomaly 2 — trend:", r2.shape, "| flagged:", r2["is_anomaly"].sum())
print("Anomaly 3 — events:", r3.shape, "| flagged:", r3["is_anomaly"].sum())

## Baseline 1 — Classical Statistical Decomposition

The oldest and most interpretable family of anomaly detection methods for time series: decompose each series into its structural components and flag observations whose residual is unexpectedly large. The intuition is that traffic follows a highly regular rhythm — daily peaks, weekly patterns, seasonal dips in August — and anything that cannot be explained by that rhythm is a candidate anomaly.

### Decomposition

The vehicle count series for each detector is decomposed additively into three components:

$$y_t = T_t + S_t + R_t$$

where $T_t$ is the **trend** (long-run evolution), $S_t$ is the **seasonal** component (multiple periods: 24 h daily + 168 h weekly), and $R_t$ is the **residual** — the part that cannot be explained by the regular structure.

Two implementations are available in `statsmodels`:

- **`seasonal_decompose`**: classical decomposition using moving averages. Simple and fast, but assumes fixed seasonality and requires a complete, evenly-spaced series.
- **`STL` (Seasonal-Trend decomposition using LOESS)**: more robust, handles gradual changes in the seasonal shape, and is less sensitive to outliers in the decomposition step itself. Preferred when the pattern evolves over the two-year observation window.

### Anomaly scoring with MAD

Once the residuals $R_t$ are extracted, a modified z-score flags outliers without being distorted by the very outliers we are trying to detect — a known weakness of the standard z-score:

$$\tilde{z}_t = \frac{0.6745 \cdot (R_t - \tilde{R})}{\text{MAD}}, \quad \text{MAD} = \text{median}(|R_t - \tilde{R}|)$$

The constant 0.6745 makes the score comparable to a standard normal z-score. A point is flagged as anomalous when $|\tilde{z}_t| > \tau$, with $\tau \in [3, 4]$ being the conventional range (Iglewicz & Hoaglin 1993).

**Why MAD instead of standard deviation?** The standard deviation is itself inflated by large anomalies, which raises the threshold and masks the very points we want to catch. The median is insensitive to outliers by construction.

### Practical notes

- Decompose **per detector**: each `chiave` has its own traffic profile and must be treated independently.
- **Exclude event days and national holidays** from the decomposition fit: a football match or a trade fair is not an anomaly in the sense of the project — it is a known, labelled disturbance. Including it in the residual distribution inflates the noise estimate.
- The decomposition requires a **continuous, regularly-sampled series** (one row per hour). Missing hours must be imputed or the series reindexed before fitting.

## Baseline 2 — Distance / Density-based Anomaly Detection

The core idea is to represent each hourly observation as a point in a feature space and flag points that are far from their neighbours or live in unusually sparse regions of that space. Unlike the statistical baseline, this approach does not decompose the series: it treats the problem as a one-class classification task where "normal" occupies a compact region and anomalies are the outliers.

### Feature engineering

Raw vehicle counts alone carry no temporal context, so each row is enriched before fitting:

| Feature | Rationale |
|---|---|
| `hour`, `dow` (day of week) | Encode the expected daily and weekly rhythm |
| `lag_1h`, `lag_24h`, `lag_168h` | Capture short-term autocorrelation and same-hour-yesterday / same-hour-last-week baselines |
| `rolling_mean_3h`, `rolling_mean_24h` | Smooth local trend; large deviations from the rolling mean are already a signal |
| `temperature_2m`, `precipitation`, `wind_speed_10m` | Weather moderates expected flow; including it prevents rain-induced drops from being flagged as anomalies |

All features are standardised (zero mean, unit variance) before fitting, since both algorithms are sensitive to scale.

### Models

**IsolationForest** (`sklearn.ensemble`) partitions the feature space by randomly splitting on random features. Anomalies require fewer splits to isolate and receive a lower anomaly score. It scales well to large datasets and is robust to the curse of dimensionality.

**LocalOutlierFactor** (`sklearn.neighbors`) compares the local density of each point to its k nearest neighbours. A point in a sparser neighbourhood than its neighbours gets a high LOF score. It is more sensitive to local structure — useful for detecting anomalies that are subtle in the global distribution but extreme in their local context (e.g. a single detector behaving oddly while nearby detectors are normal).

### Contamination rate calibration

The `contamination` parameter controls the expected fraction of anomalies and directly sets the decision threshold. Rather than defaulting to `0.01`, it should be grounded in the problem framing:

- If the project targets structural long-period anomalies (as in the three-family definition), a rate of `0.005–0.02` is defensible.
- Cross-check: the number of flagged hours should be in the same order of magnitude as known events (matches, fairs, maintenance outages) in the dataset.
- Avoid treating the contamination rate as a performance metric — it is a prior, not a result.

In [ ]:
def feat_engineering(df):
    #temporal features
    df["hour"] = df.index.hour
    df["dow"] = df.index.dayofweek #pattern of traffic by day of week
    df["lag_1h"] = df["target"].shift(1) #one hour lag
    df['lag_24h'] = df['target'].shift(24) #one day lag
    df['lag_168h'] = df['target'].shift(168) #one week lag
    df['rolling_mean_3h'] = df['target'].rolling(window=3).mean() #rolling mean of 3 hours
    df['rolling_mean_24h'] = df['target'].rolling(window=24).mean() #rolling mean of 24 hours

    return df


In [ ]:
def split(df):
    train = df[df["timestamp"] < "2025-01-01"]
    test  = df[df["timestamp"] >= "2025-01-01"]
    return train, test



In [ ]:
def isolation_forest(train, test):
    model = IsolationForest(contamination=0.01, random_state=42)
    model.fit(train)
    test = test.copy()
    test["anomaly_isolation"] = model.predict(test)
    return test



In [ ]:
def local_outlier_factor(X, df):
    model = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.01
    )

    df["anomaly_lof"] = model.fit_predict(X)

    return df

# Baseline 3

Baseline 3 — Forecasting-based. A model that predicts the expected value of a detector for a specific hour given its context (lags, day, weather) — a regressor such as GradientBoostingRegressor, HistGradientBoostingRegressor or LightGBM works well. The difference between the observed and predicted value, normalised by the model's typical error, is your anomaly score.